# 🎲 Python `random` Module — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> The `random` module is a dice factory. Inside it lives a single die with
> 2^19937 faces (Mersenne Twister). Every call to `random()` rolls it and
> converts the face number to a float between 0 and 1. All other functions
> (`randint`, `choice`, `shuffle`) are just different ways to read that same roll.
> The seed is the starting face — same seed, same sequence of faces, every time.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [What Is random? The Visual Model](#1) |
| 2 | [Importing & Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Core Float & Int Functions](#5) |
| 6 | [Pattern 2: Sequence Operations](#6) |
| 7 | [Pattern 3: Seeding & Reproducibility](#7) |
| 8 | [Pattern 4: Stress Testing & Random Pivot QuickSort](#8) |
| 9 | [Pattern 5: Reservoir Sampling (LC 382, 398)](#9) |
| 10 | [Pattern 6: secrets vs random](#10) |
| 11 | [The random Decision Map](#11) |
| 12 | [Interview Cheat Sheet](#12) |


<a id='1'></a>

## 1. What Is `random`? The Visual Model

---

```
THE MERSENNE TWISTER — ONE INTERNAL STATE, MANY OUTPUTS

  seed(42)
     │
     ▼
  ┌─────────────────────────────────────────────────────┐
  │  PRNG STATE  (624 integers, 19937 bits total)        │
  │  [2357136044, 2346248787, 3764692739, 1281271225...] │
  └─────────────────┬───────────────────────────────────┘
                    │  each call advances state + extracts one number
                    ▼
  ┌──────────────────────────────────────────────────────────────┐
  │  OUTPUT LAYER                                                │
  │                                                              │
  │  random()         →  [0.0, 1.0)  raw float                  │
  │  randint(a, b)    →  [a, b]      closed both ends           │
  │  randrange(a, b)  →  [a, b)      open right end (like range)│
  │  uniform(a, b)    →  [a, b]      float in range             │
  │  choice(seq)      →  seq[random_index]                       │
  │  choices(seq, k)  →  k draws WITH replacement               │
  │  sample(seq, k)   →  k draws WITHOUT replacement            │
  │  shuffle(lst)     →  Fisher-Yates in-place                  │
  └──────────────────────────────────────────────────────────────┘

  SEED = reset state to a known position
  ─────────────────────────────────────────────────────────────
  seed(42) → same state  → same sequence  → reproducible
  no seed  → state from OS entropy → different every run

  WHY IT MATTERS FOR INTERVIEWS:
  ┌─────────────────────────────────────────────────────────────┐
  │  stress tests:  seed → generate random input → compare     │
  │  pivot:         random.randint(lo, hi) → avoid O(n²) worst │
  │  reservoir:     random.randint(0, i)   → uniform selection  │
  └─────────────────────────────────────────────────────────────┘
```


<a id='2'></a>

## 2. Importing & Setup


In [ ]:
import random                       # standard library — no install needed
import secrets                      # crypto-safe random (covered in Pattern 6)

# The module has ONE global Random instance — all functions use it
# You can also create isolated instances (no shared state):
rng_a = random.Random(seed=42)     # private instance, seed 42
rng_b = random.Random(seed=42)     # same seed → same sequence
rng_c = random.Random(seed=99)     # different seed → different sequence

print(f"rng_a.random() = {rng_a.random():.6f}")
print(f"rng_b.random() = {rng_b.random():.6f}  (same as rng_a — same seed)")
print(f"rng_c.random() = {rng_c.random():.6f}  (different — different seed)")

# Global module functions use the shared global instance
random.seed(42)                    # reset global state to known position
print(f"random.random() = {random.random():.6f}")
print("Setup complete.")


<a id='3'></a>

## 3. The Core API — All Operations

---

```
FLOAT GENERATORS
  OPERATION               RANGE             NOTES
  ─────────────────────────────────────────────────────────────────
  random()                [0.0, 1.0)        raw roll — basis of all others
  uniform(a, b)           [a, b]            float, closed on both ends
  gauss(mu, sigma)        (-∞, +∞)          Gaussian / normal distribution
  triangular(lo, hi, mode) [lo, hi]         peak at mode

INT GENERATORS
  OPERATION               RANGE             NOTES
  ─────────────────────────────────────────────────────────────────
  randint(a, b)           [a, b]            CLOSED both ends — not like range!
  randrange(a, b)         [a, b)            open right — like range()
  randrange(a, b, step)   [a, b) step       e.g. randrange(0,10,2) → even

SEQUENCE OPERATIONS
  OPERATION               BEHAVIOR          REPLACEMENT?
  ─────────────────────────────────────────────────────────────────
  choice(seq)             one random pick   —
  choices(seq, k=n)       n picks           YES — same item can repeat
  sample(seq, k=n)        n unique picks    NO  — each item appears once max
  shuffle(lst)            in-place shuffle  mutates lst, returns None

STATE CONTROL
  OPERATION               WHAT IT DOES
  ─────────────────────────────────────────────────────────────────
  seed(x)                 reset state — reproducible from here
  getstate()              snapshot current PRNG state
  setstate(st)            restore a saved snapshot

THINGS YOU DO NOT DO:
  ❌  randint(0, n-1)  thinking it excludes n  — it INCLUDES both ends
  ❌  choice([])       on empty list           — IndexError
  ❌  sample(lst, k)   with k > len(lst)       — ValueError
  ❌  shuffle(tuple)   — tuples immutable      — TypeError
  ❌  shuffle(lst)     storing the return value — returns None
  ❌  random() for passwords/tokens           — use secrets instead
```


In [ ]:
import random
random.seed(42)    # pin state so output is reproducible in this demo

# ── Float generators ──────────────────────────────────────────────────────────
print("--- Floats ---")
print(f"random()          = {random.random():.4f}")         # [0.0, 1.0)
print(f"uniform(1.5, 6.5) = {random.uniform(1.5, 6.5):.4f}")  # [1.5, 6.5]
print(f"gauss(0, 1)       = {random.gauss(0, 1):.4f}")     # normal distribution

# ── Int generators ─────────────────────────────────────────────────────────────
print("--- Integers ---")
print(f"randint(1, 6)     = {random.randint(1, 6)}")        # die roll: 1–6 inclusive
print(f"randrange(0, 10)  = {random.randrange(0, 10)}")     # 0–9 (excludes 10)
print(f"randrange(0,10,2) = {random.randrange(0, 10, 2)}")  # even: 0,2,4,6,8

# ── Sequence operations ───────────────────────────────────────────────────────
print("--- Sequences ---")
deck = ["A", "K", "Q", "J", "10"]
print(f"choice(deck)         = {random.choice(deck)}")           # one pick
print(f"choices(deck, k=3)   = {random.choices(deck, k=3)}")     # 3 WITH replacement
print(f"sample(deck, k=3)    = {random.sample(deck, k=3)}")      # 3 WITHOUT replacement

hand = deck[:]                        # copy — shuffle mutates in-place
random.shuffle(hand)
print(f"shuffle(deck copy)   = {hand}")
print(f"shuffle returns:     = {random.shuffle(hand)}")          # None!

# ── State snapshot ────────────────────────────────────────────────────────────
print("--- State Snapshot ---")
state = random.getstate()             # save current PRNG position
r1 = random.random()
r2 = random.random()
random.setstate(state)                # rewind to saved position
r3 = random.random()
r4 = random.random()
print(f"r1={r1:.4f}, r2={r2:.4f}")
print(f"r3={r3:.4f}, r4={r4:.4f}  (same — state was restored)")
print("API demo complete.")


<a id='4'></a>

## 4. Decision Map — When To Use What

---

```
WHAT YOU WANT                          FUNCTION
─────────────────────────────────────────────────────────────────
Raw probability (0.0 to 1.0)          random()
Random float in a range               uniform(a, b)
Random integer, inclusive both ends   randint(a, b)
Random integer, exclusive right end   randrange(a, b)   ← like range()
Pick one item from a list             choice(seq)
Pick k items — repeats allowed        choices(seq, k=k)
Pick k items — no repeats             sample(seq, k=k)
Shuffle a list in-place               shuffle(lst)
Same output every run (testing)       seed(x) before any call
Replay same PRNG sequence             state = getstate(); ... setstate(state)
Crypto-safe token / password          secrets.token_hex(n) — NOT random
Stress test (generate input pairs)    seed → sample/randint → generate
Random pivot in QuickSort             randint(lo, hi) to avoid O(n²) worst
Reservoir Sampling (LC 382/398)       randint(0, i) inside loop
```


<a id='5'></a>

## 5. 🧩 Pattern 1: Core Float & Int Functions

---

```
PROBLEM:
  Know the exact range contract for each function so you never
  accidentally include/exclude a boundary in a simulation or test.

SLOW MOTION TRACE: boundary behavior
  Function            Call              Possible outputs
  ──────────────────────────────────────────────────────────
  random()            random()          0.0, 0.37, 0.99...  (never 1.0)
  uniform(1, 5)       uniform(1, 5)     1.0, 2.3, 5.0       (1.0 and 5.0 both possible)
  randint(1, 6)       randint(1, 6)     1, 2, 3, 4, 5, 6    (6 IS included!)
  randrange(0, 6)     randrange(0, 6)   0, 1, 2, 3, 4, 5    (6 NOT included)
  randrange(0, 10, 2) randrange(0,10,2) 0, 2, 4, 6, 8       (step=2, 10 NOT included)

KEY INSIGHT:
  randint is CLOSED [a,b]. randrange is OPEN [a,b). Know the difference —
  off-by-one here causes subtle simulation bugs.

TIME / SPACE:
  Time:  O(1) — single PRNG extraction
  Space: O(1) — no storage
```


In [ ]:
import random, collections

# Demonstrate boundary contracts via frequency counting
# If randint(1,6) truly includes 6, all 6 faces should appear

def verify_boundaries(n_trials=60_000):
    '''
    Boundary verification for randint vs randrange.
    Approach: run n_trials and count hits at boundary values.
    Time:  O(n_trials)
    Space: O(1)
    '''
    random.seed(0)

    # randint(1, 6) — 6-sided die, BOTH ends included
    die_counts = collections.Counter(random.randint(1, 6) for _ in range(n_trials))
    expected = n_trials / 6
    print("randint(1, 6) face frequencies:")
    for face in sorted(die_counts):
        bar = "#" * int(die_counts[face] / 200)
        print(f"  face {face}: {die_counts[face]:>6}  {bar}")

    # randrange(0, 6) — [0,5] only, 6 never appears
    range_counts = collections.Counter(random.randrange(0, 6) for _ in range(n_trials))
    print(f"
randrange(0, 6) — is 6 ever seen? {6 in range_counts}")

    # uniform(0.0, 1.0) — should hit 0.0-region and approach 1.0 but never reach exactly
    floats = [random.uniform(0, 1) for _ in range(n_trials)]
    print(f"
uniform(0, 1) min={min(floats):.6f}  max={max(floats):.6f}  mean={sum(floats)/len(floats):.4f}")

    # Simulate a biased coin: 70% heads using random()
    heads = sum(1 for _ in range(n_trials) if random.random() < 0.7)
    print(f"
Biased coin (p=0.7) heads rate: {heads/n_trials:.4f}  (expect ~0.7000)")

verify_boundaries()

# ── Practical: simulate a probability gate ────────────────────────────────────
# "With 30% chance, resample" — common in randomized algorithms
def maybe_resample(val, p_resample=0.3, rng=random):
    # roll the die — if it lands under threshold, redo
    if rng.random() < p_resample:   # [0.0, 1.0) < 0.3 → True 30% of the time
        return rng.randint(0, 100)  # new random value
    return val

random.seed(7)
for _ in range(8):
    result = maybe_resample(42)
    print(f"  maybe_resample(42) → {result}")

print("verify_boundaries defined.")


<a id='6'></a>

## 6. 🧩 Pattern 2: Sequence Operations — choice, choices, sample, shuffle

---

```
PROBLEM:
  Know when to use each sequence function — the WITH/WITHOUT replacement
  distinction is the most common interview interview gotcha.

SLOW MOTION TRACE: choices vs sample
  seq = ["A", "B", "C"]

  choices(seq, k=2) — WITH replacement — same item can appear twice:
    roll → index 0 → "A"
    roll → index 0 → "A"   ← OK, "A" again
    result = ["A", "A"]    ← valid output

  sample(seq, k=2) — WITHOUT replacement — each index used at most once:
    roll → index 2 → "C"   mark "C" used
    roll → index 0 → "A"   mark "A" used (index 2 not available now)
    result = ["C", "A"]    ← "C" can't appear again

  shuffle — Fisher-Yates algorithm:
    for i in range(len(seq)-1, 0, -1):     ← from end backwards
        j = randint(0, i)                  ← random position ≤ i
        seq[i], seq[j] = seq[j], seq[i]   ← swap
    Every permutation equally likely.      ← proof: n! outcomes, n! permutations

KEY INSIGHT:
  choices → with replacement (lottery with reset, can draw same ball twice)
  sample  → without replacement (lottery without reset, each ball drawn once)

TIME / SPACE:
  choice(seq)    O(1)    single index lookup
  choices(k)     O(k)    k rolls
  sample(k)      O(k)    k draws, rejection-sampling or partial Fisher-Yates
  shuffle(n)     O(n)    n swaps in-place
```


In [ ]:
import random

# ── choice: single random pick ────────────────────────────────────────────────
# Slow motion on choice(["N","S","E","W"]):
#   roll → random.random() = 0.37
#   index = int(0.37 * 4) = 1
#   result = "S"
directions = ["N", "S", "E", "W"]
random.seed(1)
picks = [random.choice(directions) for _ in range(8)]
print(f"choice x8: {picks}")

# ── choices vs sample ─────────────────────────────────────────────────────────
# choices: with replacement — lottery where you put the ball back
# sample:  without replacement — lottery where ball stays out
pool = list(range(1, 11))   # [1..10]
random.seed(5)

with_rep = random.choices(pool, k=5)    # 5 draws, repeats allowed
no_rep   = random.sample(pool, k=5)     # 5 draws, all unique

print(f"choices(1..10, k=5): {sorted(with_rep)}  (repeats possible: {len(set(with_rep)) < 5})")
print(f"sample(1..10, k=5):  {sorted(no_rep)}   (repeats: {len(set(no_rep)) < 5})")

# ── weighted choices ─────────────────────────────────────────────────────────
# choices() supports weights= — simulate a loaded die
faces  = [1,   2,   3,   4,   5,   6  ]
weights = [10, 10,  10,  10,  10,  50 ]   # face 6 is 5x more likely

random.seed(0)
rolls = random.choices(faces, weights=weights, k=12_000)
from collections import Counter
freq = Counter(rolls)
print("
Loaded die (face 6 is 5x heavier):")
for f in sorted(freq):
    pct = freq[f] / 120
    bar = "#" * int(pct)
    print(f"  face {f}: {pct:.1f}%  {bar}")

# ── Fisher-Yates shuffle — manual trace ──────────────────────────────────────
# Slow motion on [0, 1, 2, 3]:
# i=3: j=randint(0,3)=1  →  swap [3] with [1]  →  [0, 3, 2, 1]
# i=2: j=randint(0,2)=0  →  swap [2] with [0]  →  [2, 3, 0, 1]
# i=1: j=randint(0,1)=1  →  swap [1] with [1]  →  [2, 3, 0, 1]  (noop)
# result: [2, 3, 0, 1]
lst = [0, 1, 2, 3]
random.seed(99)
random.shuffle(lst)
print(f"
After shuffle([0,1,2,3]): {lst}")

# Verify all permutations equally likely over many shuffles
from itertools import permutations
base = [1, 2, 3]
perm_counts = Counter()
for _ in range(60_000):
    copy = base[:]
    random.shuffle(copy)
    perm_counts[tuple(copy)] += 1

print("
Shuffle uniformity (expect ~10000 each):")
for perm, cnt in sorted(perm_counts.items()):
    bar = "#" * (cnt // 1000)
    print(f"  {perm}: {cnt}  {bar}")

print("Sequence operations complete.")


<a id='7'></a>

## 7. 🧩 Pattern 3: Seeding & Reproducibility

---

```
PROBLEM:
  Tests and experiments need to produce the same output every run.
  Without seeding, random tests are non-deterministic — a passing test
  might fail next run on a different sequence.

SLOW MOTION TRACE: two runs, same seed
  Run 1:  seed(42) → random() = 0.6394  → randint(0,9) = 1
  Run 2:  seed(42) → random() = 0.6394  → randint(0,9) = 1
  Same. Always.

  Run 3:  no seed  → random() = 0.4827  → randint(0,9) = 7  (OS entropy)
  Run 4:  no seed  → random() = 0.1931  → randint(0,9) = 3  (different)

getstate / setstate — snapshot any PRNG position:
  state = random.getstate()   # save current position
  seq1 = [random.randint(0,9) for _ in range(5)]
  random.setstate(state)      # rewind
  seq2 = [random.randint(0,9) for _ in range(5)]
  seq1 == seq2   → True       # same sequence replayed

USE CASES:
  ✅ stress test: seed → generate expected input → compare two algos
  ✅ ML shuffle: seed before train/test split → reproducible splits
  ✅ debug: seed before reproducing a failing random case
  ✅ isolated instance: random.Random(seed) for thread-safe private PRNG

KEY INSIGHT:
  seed(x) is a time machine for the PRNG — it teleports the die
  back to face x, guaranteeing the next n rolls are identical.

TIME / SPACE:
  Time:  O(1) for seed(), O(state_size) for get/setstate
  Space: O(1) for seed(), O(624 ints) = O(1) fixed for getstate
```


In [ ]:
import random

# ── seed() makes outputs identical across runs ────────────────────────────────
def generate_test_input(n, lo=0, hi=100, seed_val=42):
    '''
    Reproducible random array for stress testing.
    Approach: fix seed → generate → same array every call.
    Time:  O(n)
    Space: O(n)
    '''
    random.seed(seed_val)              # teleport PRNG to known state
    return [random.randint(lo, hi) for _ in range(n)]

arr1 = generate_test_input(8)
arr2 = generate_test_input(8)          # same seed → same array
arr3 = generate_test_input(8, seed_val=99)  # different seed → different array

print(f"seed=42: {arr1}")
print(f"seed=42: {arr2}  (identical: {arr1 == arr2})")
print(f"seed=99: {arr3}  (different: {arr1 != arr3})")

# ── getstate / setstate — replay any sequence ─────────────────────────────────
random.seed(0)
_ = [random.random() for _ in range(100)]   # advance state some amount
checkpoint = random.getstate()              # bookmark this position

# Slow motion: state saved, advance, rewind, advance again → same sequence
seq_a = [random.randint(0, 9) for _ in range(6)]
random.setstate(checkpoint)                 # rewind to bookmark
seq_b = [random.randint(0, 9) for _ in range(6)]
print(f"
seq_a: {seq_a}")
print(f"seq_b: {seq_b}  (replayed: {seq_a == seq_b})")

# ── Isolated Random instance — thread-safe, independent state ─────────────────
# Two threads each with their own PRNG — no shared state conflicts
rng_test   = random.Random(42)   # for test generation
rng_algo   = random.Random(99)   # for algorithm internals

test_arr  = [rng_test.randint(0, 50) for _ in range(5)]
algo_pick = rng_algo.randint(0, len(test_arr) - 1)
print(f"
test array:  {test_arr}")
print(f"algo picked: index={algo_pick}, value={test_arr[algo_pick]}")

# ── Stress test harness skeleton ──────────────────────────────────────────────
def stress_test(brute_fn, fast_fn, n_cases=200, n=10):
    '''
    Compare brute_fn vs fast_fn on n_cases random inputs.
    Fails fast on first mismatch with the exact input that broke it.
    '''
    for case_id in range(n_cases):
        random.seed(case_id)          # each case is reproducible by case_id
        arr = [random.randint(0, 20) for _ in range(n)]
        target = random.randint(0, 40)

        expected = brute_fn(arr, target)
        got      = fast_fn(arr, target)
        if expected != got:
            print(f"MISMATCH case {case_id}: arr={arr} target={target}")
            print(f"  brute={expected}  fast={got}")
            return
    print(f"{n_cases}/{n_cases} stress test cases passed.")

# Demo: two identical impls (both pass)
def count_ge_brute(arr, target):
    return sum(1 for x in arr if x >= target)   # O(n) naive

def count_ge_fast(arr, target):
    return sum(1 for x in arr if x >= target)   # same for demo

stress_test(count_ge_brute, count_ge_fast)
print("Seeding demo complete.")


<a id='8'></a>

## 8. 🧩 Pattern 4: Stress Testing & Random Pivot QuickSort

---

```
PROBLEM:
  QuickSort with a fixed pivot (always first or last element) degrades to
  O(n²) on sorted or nearly-sorted input — a common interview gotcha.
  Randomizing the pivot gives expected O(n log n) regardless of input.

TRICK:
  Before partitioning, swap a random element into the pivot position.
  The adversary can no longer craft worst-case input because they don't
  know which element will be chosen.

SLOW MOTION TRACE: random pivot on [3, 1, 4, 1, 5, 9, 2, 6]  lo=0 hi=7
  step 1: pivot_idx = randint(0, 7) = 4   →  arr[4]=5
  step 2: swap arr[4] with arr[hi=7]      →  arr = [3,1,4,1,6,9,2,5]
  step 3: pivot = arr[hi] = 5  (Lomuto partition)
  step 4: partition → [3,1,4,1,2, | 5, | 6,9]
  step 5: recurse on [3,1,4,1,2] and [6,9]
  (each step the pivot is random — no adversarial input wins)

KEY INSIGHT:
  O(n²) worst case for QuickSort only happens when pivot is always
  min or max. Random pivot → expected O(n log n) — the bad case
  has probability → 0 as n grows.

TIME / SPACE:
  Time:  O(n log n) expected  — random pivot prevents O(n²) worst case
  Space: O(log n) expected    — call stack depth; O(n) worst case stack
```


In [ ]:
import random, time

def quicksort_random_pivot(arr, lo=0, hi=None):
    '''
    QuickSort with randomized pivot — avoids O(n²) worst case.
    Approach: swap random element to hi, then Lomuto partition.
    Args:
        arr (list): mutable list to sort in-place.
        lo  (int): left boundary (inclusive).
        hi  (int): right boundary (inclusive), defaults to len-1.
    Returns:
        None — sorts arr in-place.
    Time:  O(n log n) expected — random pivot prevents adversarial O(n²)
    Space: O(log n) expected  — recursion depth on random splits
    '''
    if hi is None:
        hi = len(arr) - 1

    if lo >= hi:
        return   # base case: zero or one element — already sorted

    # Slow motion on arr=[3,1,4,1,5,9,2,6] lo=0 hi=7:
    # pivot_idx = randint(0, 7)           →  say 4
    # swap arr[4]=5 with arr[hi=7]=6      →  arr=[3,1,4,1,6,9,2,5]
    # pivot = arr[hi] = 5 (Lomuto)
    # i=-1 (wall between ≤ and > pivot)
    # j=0: arr[0]=3 < 5  →  i=0, swap arr[0]↔arr[0]  →  wall moves right
    # j=1: arr[1]=1 < 5  →  i=1, swap arr[1]↔arr[1]
    # j=2: arr[2]=4 < 5  →  i=2, swap arr[2]↔arr[2]
    # j=3: arr[3]=1 < 5  →  i=3, swap arr[3]↔arr[3]
    # j=4: arr[4]=6 > 5  →  skip
    # j=5: arr[5]=9 > 5  →  skip
    # j=6: arr[6]=2 < 5  →  i=4, swap arr[4]=6 with arr[6]=2
    # place pivot: swap arr[i+1=5] with arr[hi=7]
    # arr=[3,1,4,1,2, | 5, | 9,6]   pivot 5 is now in final position

    pivot_idx = random.randint(lo, hi)    # random pivot — adversary can't predict
    arr[pivot_idx], arr[hi] = arr[hi], arr[pivot_idx]  # move pivot to end

    pivot = arr[hi]
    i = lo - 1                            # i = wall: everything left of i is ≤ pivot

    for j in range(lo, hi):              # j scans the unexamined region
        if arr[j] <= pivot:
            i += 1                        # expand the ≤ region
            arr[i], arr[j] = arr[j], arr[i]  # swap current element into ≤ region

    pivot_pos = i + 1                     # pivot goes right after the ≤ region
    arr[pivot_pos], arr[hi] = arr[hi], arr[pivot_pos]

    quicksort_random_pivot(arr, lo, pivot_pos - 1)   # recurse left of pivot
    quicksort_random_pivot(arr, pivot_pos + 1, hi)   # recurse right of pivot


def test_harness(fn):
    tests = [
        ([3, 1, 4, 1, 5, 9, 2, 6], [1, 1, 2, 3, 4, 5, 6, 9]),
        ([], []),
        ([1], [1]),
        ([2, 1], [1, 2]),
        (list(range(20, 0, -1)), list(range(1, 21))),   # reverse sorted = old O(n²) case
        ([5]*10, [5]*10),                               # all equal
    ]
    passed = 0
    for *inputs, expected in tests:
        arr = inputs[0][:]                  # copy — sort is in-place
        fn(arr)
        got = arr
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

random.seed(42)
test_harness(quicksort_random_pivot)

# ── Timing: random pivot vs fixed-last on sorted input ────────────────────────
def quicksort_bad_pivot(arr, lo=0, hi=None):
    # Fixed pivot at hi — O(n²) on sorted input
    if hi is None: hi = len(arr) - 1
    if lo >= hi: return
    pivot = arr[hi]
    i = lo - 1
    for j in range(lo, hi):
        if arr[j] <= pivot:
            i += 1; arr[i], arr[j] = arr[j], arr[i]
    arr[i+1], arr[hi] = arr[hi], arr[i+1]
    quicksort_bad_pivot(arr, lo, i)
    quicksort_bad_pivot(arr, i+2, hi)

import sys
sys.setrecursionlimit(10_000)
n = 1000
sorted_input = list(range(n))

arr1 = sorted_input[:]
start = time.perf_counter(); random.seed(0); quicksort_random_pivot(arr1)
t_rand = time.perf_counter() - start

arr2 = sorted_input[:]
start = time.perf_counter(); quicksort_bad_pivot(arr2)
t_bad = time.perf_counter() - start

print(f"
n={n} sorted input:")
print(f"  random pivot: {t_rand*1000:.2f}ms")
print(f"  fixed pivot:  {t_bad*1000:.2f}ms  (~{t_bad/t_rand:.0f}x slower)")
print("quicksort_random_pivot defined.")


<a id='9'></a>

## 9. 🧩 Pattern 5: Reservoir Sampling — LC 382, LC 398

---

```
PROBLEM LC 382:
  Linked list of unknown length n. Pick one node uniformly at random
  using O(1) space. You cannot store the list.

TRICK — Reservoir Sampling (k=1):
  Process each node i (0-indexed). Keep current result.
  At node i, replace result with node i with probability 1/(i+1).

SLOW MOTION TRACE on [3, 1, 4, 1, 5, 9]:
  i=0: node=3  p=1/1=1.00  →  always pick  →  result=3
  i=1: node=1  p=1/2=0.50  →  roll > 0.5   →  result=3  (kept)
  i=2: node=4  p=1/3=0.33  →  roll > 0.33  →  result=3  (kept)
  i=3: node=1  p=1/4=0.25  →  roll < 0.25  →  result=1  (replaced!)
  i=4: node=5  p=1/5=0.20  →  roll > 0.2   →  result=1  (kept)
  i=5: node=9  p=1/6=0.17  →  roll > 0.17  →  result=1  (final)

WHY IT'S UNIFORM:
  P(element i is in reservoir at the end)
  = P(i was picked) × P(i not evicted by i+1) × ... × P(i not evicted by n-1)
  = (1/i+1) × (i+1/i+2) × (i+2/i+3) × ... × (n-1/n)
  = 1/n     ← telescoping product — every element equally likely!

KEY INSIGHT:
  You don't know n. That's the point. Reservoir sampling gives uniform
  probability using only ONE pass and O(1) memory (for k=1 reservoir).

TIME / SPACE:
  Time:  O(n) — one pass through the linked list
  Space: O(1) — only the current result stored (k=1 case)
```


In [ ]:
import random

# ── ListNode helper ───────────────────────────────────────────────────────────
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

class SolutionLC382:
    '''
    LC 382 — Linked List Random Node.
    Approach: Reservoir Sampling, k=1.
    Time:  O(n) per getRandom call — full pass
    Space: O(1) — only head pointer + current result stored
    '''
    def __init__(self, head: ListNode):
        self.head = head   # store only the head — O(1) space

    def getRandom(self) -> int:
        # Slow motion on head → [3, 1, 4, 1, 5]:
        # i=0: result=3,  replace with p=1/(0+1)=1.0  → result=3
        # i=1: result=3,  replace with p=1/(1+1)=0.5  → maybe result=1
        # i=2: result=?,  replace with p=1/(2+1)=0.33 → maybe result=4
        # ...
        result = None
        node = self.head
        i = 0
        while node:
            if random.randint(0, i) == 0:   # probability = 1/(i+1)
                result = node.val           # replace reservoir with current node
            node = node.next
            i += 1
        return result


class SolutionLC398:
    '''
    LC 398 — Random Pick Index.
    Approach: Reservoir Sampling over indices matching target.
    Time:  O(n) per pick call
    Space: O(n) — stores nums array (could stream, but simpler to store)
    '''
    def __init__(self, nums: list):
        self.nums = nums

    def pick(self, target: int) -> int:
        # Only consider indices where nums[i] == target
        # Among those, pick uniformly using reservoir sampling
        result = -1
        count = 0                          # how many matching indices seen so far
        for i, val in enumerate(self.nums):
            if val == target:
                count += 1
                if random.randint(1, count) == 1:  # 1/count probability
                    result = i             # replace reservoir
        return result


# ── Verify uniformity of reservoir sampling ───────────────────────────────────
def verify_reservoir_uniform(vals, n_trials=60_000, seed=0):
    '''Confirm each index is chosen ~1/n of the time.'''
    from collections import Counter
    head = make_list(vals)
    sol = SolutionLC382(head)
    random.seed(seed)
    counts = Counter(sol.getRandom() for _ in range(n_trials))
    expected = n_trials / len(vals)
    print(f"Reservoir sampling on {vals} ({n_trials} trials):")
    for val in vals:
        pct = counts[val] / n_trials * 100
        bar = "#" * int(pct / 2)
        print(f"  val={val}: {counts[val]:>6} ({pct:.1f}%)  {bar}")
    print(f"  Expected: {expected:.0f} each ({100/len(vals):.1f}%)")

verify_reservoir_uniform([3, 1, 4, 1, 5, 9])

# ── LC 398: pick index with duplicates ────────────────────────────────────────
random.seed(42)
sol398 = SolutionLC398([1, 2, 3, 3, 3])
from collections import Counter
picks = Counter(sol398.pick(3) for _ in range(30_000))
print(f"
LC 398: pick(3) from [1,2,3,3,3] — indices 2,3,4 should be ~equal:")
for idx in sorted(picks):
    print(f"  index {idx}: {picks[idx]} ({picks[idx]/300:.1f}%)")

print("SolutionLC382 + SolutionLC398 defined.")


<a id='10'></a>

## 10. 🧩 Pattern 6: `secrets` vs `random` — Crypto-Safe vs Simulation

---

```
PROBLEM:
  random is NOT cryptographically secure. Its output is predictable if
  you know the seed or observe enough outputs. Use secrets for anything
  security-sensitive.

random MODULE:
  ✅  Simulations, stress tests, games, ML shuffles
  ✅  Random pivots in algorithms
  ✅  Test data generation
  ❌  Passwords, tokens, session IDs, OTPs
  ❌  Any case where an attacker might try to predict the output

secrets MODULE (Python 3.6+):
  ✅  Cryptographically secure (uses OS entropy: /dev/urandom or CryptGenRandom)
  ✅  Passwords, tokens, session IDs, reset links, OTPs
  ❌  Slow for bulk random data (not for simulations)
  ❌  Not reproducible (no seed)

COMPARISON TABLE:
  Feature               random          secrets
  ────────────────────────────────────────────────────────
  Source                Mersenne Twister  OS entropy
  Reproducible?         YES (seed)        NO
  Cryptographically safe? NO              YES
  Speed                 Fast              Slower
  Use in LC/interviews  YES               Rarely
  Use in prod systems   Simulations only  Auth/tokens

KEY INSIGHT:
  If you'd be embarrassed if an attacker guessed it, use secrets.
  If you just need "good enough random for testing", use random.

TIME / SPACE:
  secrets.token_hex(n)  O(n) — generates n bytes, returns 2n hex chars
  secrets.randbelow(n)  O(1) — secure integer in [0, n)
```


In [ ]:
import random, secrets, time

# ── random: fast, seeded, predictable ─────────────────────────────────────────
random.seed(42)
r_token = "".join(str(random.randint(0, 9)) for _ in range(16))
print(f"random token:     {r_token}  ← reproducible! seed(42) always gives this")

random.seed(42)
r_token2 = "".join(str(random.randint(0, 9)) for _ in range(16))
print(f"random token:     {r_token2}  ← exact same — attacker can compute this")

# ── secrets: slow, non-reproducible, safe ────────────────────────────────────
s_hex   = secrets.token_hex(16)         # 32-char hex string from 16 random bytes
s_url   = secrets.token_urlsafe(16)     # URL-safe base64 from 16 bytes
s_bytes = secrets.token_bytes(8)        # raw random bytes

print(f"
secrets.token_hex(16):      {s_hex}")
print(f"secrets.token_urlsafe(16):  {s_url}")
print(f"secrets.token_bytes(8):     {s_bytes.hex()}")

# Generate again — always different
s_hex2 = secrets.token_hex(16)
print(f"secrets.token_hex(16):      {s_hex2}  ← different every call")

# ── secrets.randbelow — uniform secure int ────────────────────────────────────
otp = "".join(str(secrets.randbelow(10)) for _ in range(6))
print(f"
6-digit OTP (secrets): {otp}")

# ── Speed comparison ──────────────────────────────────────────────────────────
N = 10_000

start = time.perf_counter()
for _ in range(N):
    random.randint(0, 2**32)
t_random = time.perf_counter() - start

start = time.perf_counter()
for _ in range(N):
    secrets.randbelow(2**32)
t_secrets = time.perf_counter() - start

print(f"
Speed comparison ({N} random ints):")
print(f"  random.randint:      {t_random*1000:.2f}ms")
print(f"  secrets.randbelow:   {t_secrets*1000:.2f}ms  ({t_secrets/t_random:.1f}x slower)")
print("
Conclusion: random is faster — fine for simulations. secrets for auth tokens.")

# ── Decision: which to use ────────────────────────────────────────────────────
examples = [
    ("Shuffle test data for ML training",          "random.shuffle(data)"),
    ("Password reset token (email link)",          "secrets.token_urlsafe(32)"),
    ("Random pivot in QuickSort",                  "random.randint(lo, hi)"),
    ("Session cookie / CSRF token",                "secrets.token_hex(32)"),
    ("Stress test random input generation",        "random.seed(x); random.sample(...)"),
    ("2FA one-time password",                      "secrets.randbelow(1_000_000)"),
]
print("
Decision table:")
for use_case, solution in examples:
    print(f"  {use_case:<45}  {solution}")

print("secrets vs random demo complete.")


<a id='11'></a>

## 11. The `random` Decision Map

---

```
QUESTION TYPE                         KEY TECHNIQUE             FUNCTIONS
────────────────────────────────────────────────────────────────────────────────
Need a float 0.0 to 1.0              Raw PRNG roll             random()
Need a float in range [a,b]          Scaled roll               uniform(a,b)
Need an int, BOTH ends included      Closed integer            randint(a,b)
Need an int, open right end          Half-open integer         randrange(a,b)
Pick one item from sequence          Single draw               choice(seq)
Pick k items, repeats OK             Draw with replacement     choices(seq,k=k)
Pick k unique items                  Draw without replacement  sample(seq,k=k)
Randomize a list in-place            Fisher-Yates              shuffle(lst)
Same output every run                Pin the die face          seed(x)
Replay a PRNG sequence               Snapshot state            getstate/setstate
Isolate PRNG from other code         Private instance          random.Random(seed)
Avoid QuickSort O(n²) worst case     Random pivot              randint(lo,hi)
Uniform pick from unknown-length stream  Reservoir sampling    randint(0,i)  LC 382
Uniform pick by value w/ duplicates  Reservoir by target       LC 398
Crypto-safe token / password         OS entropy                secrets.token_*
Stress test: reproducible random input  seed per case          seed(case_id)
────────────────────────────────────────────────────────────────────────────────

BOUNDARY CONTRACTS (memorize):
  randint(a, b)     → [a, b]     CLOSED both ends   ← NOT like range!
  randrange(a, b)   → [a, b)     open right end     ← LIKE range
  uniform(a, b)     → [a, b]     CLOSED both ends (float)
  random()          → [0.0, 1.0) open right end     ← never exactly 1.0
```


<a id='12'></a>

## 12. Interview Cheat Sheet

---

**1. When to reach for `random`:**

| Signal | What to Do |
|--------|-----------|
| "stress test your solution" | `seed(case_id)` + `randint`/`sample` |
| "QuickSort" + sorted input risk | random pivot: `randint(lo, hi)` |
| "pick from stream of unknown length" | Reservoir Sampling |
| "reproducible test" | `seed(42)` before any generation |
| "shuffle training data" | `shuffle(data)` after `seed(x)` |
| "generate random password/token" | use `secrets`, not `random` |

**2. The key operations — memorize these:**

```python
import random

random.seed(42)                      # pin state — reproducible
random.random()                      # [0.0, 1.0) raw float
random.uniform(a, b)                 # [a, b] float
random.randint(a, b)                 # [a, b] CLOSED int — includes b
random.randrange(a, b)               # [a, b) open int — excludes b
random.choice(seq)                   # one random element
random.choices(seq, k=n)             # n elements WITH replacement
random.sample(seq, k=n)              # n elements WITHOUT replacement
random.shuffle(lst)                  # in-place, returns None

# State snapshot
state = random.getstate()
random.setstate(state)

# Private instance (thread-safe, isolated)
rng = random.Random(42)
rng.randint(0, 100)
```

**3. Common templates:**

```python
# TEMPLATE: Stress test harness
def stress_test(brute, fast, n_cases=500, n=10):
    for case_id in range(n_cases):
        random.seed(case_id)
        arr = [random.randint(0, 50) for _ in range(n)]
        target = random.randint(0, 100)
        if brute(arr, target) != fast(arr, target):
            print(f"FAIL case {case_id}: arr={arr}")
            return
    print("All stress tests passed.")

# TEMPLATE: Random pivot QuickSort
def partition_random(arr, lo, hi):
    idx = random.randint(lo, hi)
    arr[idx], arr[hi] = arr[hi], arr[idx]   # pivot to end
    pivot = arr[hi]
    i = lo - 1
    for j in range(lo, hi):
        if arr[j] <= pivot:
            i += 1; arr[i], arr[j] = arr[j], arr[i]
    arr[i+1], arr[hi] = arr[hi], arr[i+1]
    return i + 1

# TEMPLATE: Reservoir Sampling k=1 (LC 382)
def reservoir_pick(head):
    result, i = None, 0
    node = head
    while node:
        if random.randint(0, i) == 0:   # prob 1/(i+1)
            result = node.val
        node = node.next; i += 1
    return result

# TEMPLATE: Secure token (NOT random module)
import secrets
token = secrets.token_urlsafe(32)   # 32-byte URL-safe token for session IDs
otp   = secrets.randbelow(1_000_000)  # 6-digit secure OTP
```

**4. Gotchas:**

```
❌  randint(0, n-1)  thinking it excludes n — randint INCLUDES both ends
❌  randrange(0, n)  thinking it includes n — it doesn't (like range)
❌  shuffle(lst)  storing the return value — returns None
❌  choice([])  — IndexError on empty sequence
❌  sample(seq, k)  with k > len(seq) — ValueError
❌  random() for passwords — use secrets
❌  choices() when you need unique picks — use sample()
✅  seed(case_id) per stress-test case — reproducible and independent
✅  random.Random(seed) for thread-safe independent PRNG instances
✅  reservoir sampling when list length is unknown
✅  randint(lo, hi) for random pivot before partitioning
```


```
                    🎲 random MODULE MAP

                    import random
                         │
           ┌─────────────┼──────────────────┐
           ▼             ▼                  ▼
        FLOATS         INTEGERS          SEQUENCES
        ──────         ────────          ─────────
        random()       randint(a,b)      choice(seq)
        uniform(a,b)   randrange(a,b)    choices(seq, k, weights)
        gauss(μ,σ)     randrange(a,b,step) sample(seq, k)
                                         shuffle(lst)

        REPRODUCIBILITY          INTERVIEW PATTERNS
        ───────────────          ──────────────────
        seed(x)                  Random Pivot QuickSort
        getstate()                 randint(lo, hi) before partition
        setstate(st)             Reservoir Sampling (LC 382/398)
        Random(seed) instance      randint(0, i) — prob 1/(i+1)
                                 Stress Testing
                                   seed(case_id) per test

        SECURITY BOUNDARY
        ─────────────────
        random  → simulation, tests, algorithms   (fast, predictable)
        secrets → tokens, passwords, OTPs         (OS entropy, safe)

        BOUNDARY CONTRACTS
        ──────────────────
        randint(a,b)   [a, b]    ← CLOSED  (includes b)
        randrange(a,b) [a, b)    ← OPEN    (excludes b, like range)
        uniform(a,b)   [a, b]    ← CLOSED  (float)
        random()       [0.0,1.0) ← OPEN    (never exactly 1.0)

---
*End of random Module Master Guide — Sean Edition*
```
